In [1]:
import pandas as pd


In [2]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'tip_amount', 'total_amount', ]
df = pd.read_parquet(url, columns=columns)
df.head()


,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [3]:
len(df)

49416

In [4]:
row = df.iloc[0]
row 

lpep_pickup_datetime     2025-10-01 00:21:47
lpep_dropoff_datetime    2025-10-01 00:24:37
PULocationID                             247
DOLocationID                              69
passenger_count                          1.0
trip_distance                            0.7
tip_amount                               1.7
total_amount                            10.0
Name: 0, dtype: object

In [5]:
from models_green import GreenRide, ride_from_row, ride_serializer


In [6]:
ride = ride_from_row(df.iloc[0])
ride
# Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72,
#      total_amount=17.31, tpep_pickup_datetime=1730429702000)

GreenRide(lpep_pickup_datetime=1759278107000, lpep_dropoff_datetime=1759278277000, string_lpep_pickup_datetime='2025-10-01 00:21:47', string_lpep_dropoff_datetime='2025-10-01 00:24:37', PULocationID=247, DOLocationID=69, passenger_count=1, trip_distance=0.7, tip_amount=1.7, total_amount=10.0)

In [7]:
from kafka import KafkaProducer

server = 'localhost:9092'
topic_name = 'green-trips'


In [8]:
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [8]:
producer.send(topic_name, value=ride)
producer.flush()

In [9]:
import time

t0 = time.time()

for i, (_, row) in enumerate(df.iterrows(), start=1):
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)

    if i % 1000 == 0:
        print(f"Sent: {ride}")      

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

Sent: GreenRide(lpep_pickup_datetime=1759340063000, lpep_dropoff_datetime=1759340693000, string_lpep_pickup_datetime='2025-10-01 17:34:23', string_lpep_dropoff_datetime='2025-10-01 17:44:53', PULocationID=82, DOLocationID=138, passenger_count=1, trip_distance=3.62, tip_amount=5.34, total_amount=32.04)
Sent: GreenRide(lpep_pickup_datetime=1759413404000, lpep_dropoff_datetime=1759414376000, string_lpep_pickup_datetime='2025-10-02 13:56:44', string_lpep_dropoff_datetime='2025-10-02 14:12:56', PULocationID=74, DOLocationID=236, passenger_count=1, trip_distance=2.19, tip_amount=4.11, total_amount=24.66)
Sent: GreenRide(lpep_pickup_datetime=1759477922000, lpep_dropoff_datetime=1759480550000, string_lpep_pickup_datetime='2025-10-03 07:52:02', string_lpep_dropoff_datetime='2025-10-03 08:35:50', PULocationID=92, DOLocationID=202, passenger_count=1, trip_distance=8.41, tip_amount=2.0, total_amount=47.8)
Sent: GreenRide(lpep_pickup_datetime=1759519440000, lpep_dropoff_datetime=1759521444000, stri